# Notebook for initial GRU implementation

In [16]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0,str(ROOT))

from carfollow import (load_table, split_by_group, iter_pairs, HISTORY_FEATURES, HISTORY_STEPS, history_at)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

ROOT

PosixPath('/Users/l5s/Desktop/UT - Projects/DSE 511/dse511-car-following')

## Read in the data using imported functions

In [17]:
# reading data in and creating the trainning and test split by group number using split_by_group
data_df = load_table()
train_val , test_df = split_by_group(data_df, test_group=0)
train_df, val_df = split_by_group(train_val, test_group=1)

# Making sure the groups split upt correctly
print("Groups", test_df['driver_group'].unique(), "Length: ", len(test_df))
print("Groups", val_df['driver_group'].unique(), "Length: ", len(val_df))
print("Groups", train_df['driver_group'].unique(), "Length: ", len(train_df))

assert len(data_df) == (len(test_df) + len(train_df) + len(val_df))

Groups [0] Length:  16676
Groups [1] Length:  15212
Groups [4 3 2] Length:  48493


In [ ]:
# Creating windows to feed into model --> Looking back 3 seconds
def make_windows(df, lookback_steps=HISTORY_STEPS, feature_cols=HISTORY_FEATURES):
    """
    Slicing every pair into windows of lookback_steps rows

    Returns:
        X: array of n windows, with look_backsteps, 3 (# num features)
        Y: array of target accel_mps2 at each window's last row
    """
    X, y = [], []
    for pair in iter_pairs(df):
        
        # Putting raw values into numpy arrays for efficiency  --> float 32
        features = pair[list(feature_cols)].to_numpy(np.float32) 
        acceleration = pair["accel_mps2"].to_numpy(np.float32)

        # Every row in data represents 0.1s of time passing
        # Starting at 29 --> 30 rows before prediction
        for i in range(lookback_steps - 1, len(pair) - 1):
            # Grab the rows
            window = features[i - lookback_steps + 1: i + 1]
            X.append(window)
            # Target acceleration
            y.append(acceleration[i])
    return np.stack(X), np.array(y)

In [ ]:
X_train, y_train = make_windows(train_df)
X_val, y_val = make_windows(val_df)
X_test, y_test = make_windows(test_df)

# Checking shape
print(X_train.shape, X_test.shape, X_test.shape)
print(y_train.shape, y_test.shape, y_test.shape)

(44233, 30, 3) (15266, 30, 3) (15266, 30, 3)
(44233,) (15266,) (15266,)


In [34]:
BATCH_SIZE = 265

# Need to normalize values before feeding to the GRU model
# Currently Gap is in meters and has much larger range than both speed and closing speed in m/s
mean = X_train.reshape(-1, 3).mean(0)
std = X_train.reshape(-1,3).std(0)

print(mean.round(3),  std.round(3))


def load_data(X, y, shuffle):
    """
    normalizes and uses dataloader to prepare data for model
    """
    X_normalized = torch.tensor((X - mean) / std, dtype=torch.float32)
    return DataLoader(TensorDataset(X_normalized, torch.tensor(y)), batch_size=BATCH_SIZE, shuffle=shuffle)

train_dl = load_data(X_train, y_train, True)
val_dl = load_data(X_val, y_val, False)
test_dl = load_data(X_test, y_test, False)

# Looking at shapes
batch = next(iter(train_dl))
x , y = batch
print(x.shape, y.shape)

[20.074 12.543 -0.396] [10.95   3.78   1.271]
torch.Size([265, 30, 3]) torch.Size([265])


In [35]:
HIDDEN = 32

class GRUFollower(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.gru = nn.GRU(input_size=3, hidden_size=HIDDEN, batch_first=True)
        # making 1 prediction
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:,-1].squeeze(-1))

model = GRUFollower()